<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 5: Tool Wear Classification and Model Performance Evaluation</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week05/lab05_tool_wear_classification.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

### ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University**

**Graded individual Lab · Approximately 90 minutes**

**Version:** Student Notebook

Open in Colab and save a personal copy before editing. Canvas controls submission requirements and due dates.

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview

Classify tool wear using the same sensor features as Lab 4: first Normal/Severe,
then Slight/Intermediate/Severe.

By the end, you can:
- fit Logistic Regression and SVM classifiers;
- interpret confusion matrices, precision, recall and F1;
- select a model using validation F1 and explain its test results.

**Your work:** five small code edits and four brief checkpoints. Run provided code
unchanged. Record your choices before checking the results.

On your first pass, run cells in order and complete each checkpoint before
continuing. Use **Run all** only after completing your work, to verify the saved notebook.

For provided code, understand its purpose and read its outputs; you do not need to memorize every setup or plotting command. Replace only the value or expression beside each TODO, keeping its variable name. After changing a cell, rerun it and the cells below it.

## Data

Run this Lab in **Google Colab**. The next two cells import the tools and load
the PHM feature CSV from the course repository; no data upload is needed.
As in Lab 4, each row is one cut, and mean wear averages three flute measurements (µm).
[Dataset details and source](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md).

![Instructor-provided PHM measurement-to-classification overview](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab05/phm_classification_overview.png)

In [ ]:
# PROVIDED — RUN UNCHANGED. Imports do not fit a model.
import numpy as np  # Compare arrays of labels during the data check.
import pandas as pd  # Read CSV files and work with tables.
import matplotlib.pyplot as plt  # Draw figures.
from IPython.display import display  # Show a table in the notebook.
# Import each tool needed to split data, scale inputs and train classifiers.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

plt.rcParams['font.size'] = 11  # Set the default text size in plots.
plt.rcParams['figure.dpi'] = 110  # Set the plot display resolution.

In [ ]:
# PROVIDED — RUN UNCHANGED. Load the course CSV directly in Colab.
data_url = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv'
data = pd.read_csv(data_url)  # Internet access is required; no file selection.

# Detect an older CSV rather than silently using the earlier 50 µm boundary.
# Start a separate check column; do not overwrite the supplied labels.
expected_level = pd.Series(0, index=data.index)
# Values above 75 micrometers are at least Intermediate-wear.
expected_level.loc[data['wear_mean_um'] > 75] = 1
# Values at or above 150 micrometers belong to Severe-wear instead.
expected_level.loc[data['wear_mean_um'] >= 150] = 2
if not np.array_equal(data['wear_level'], expected_level):
    raise ValueError('This CSV has outdated/inconsistent wear labels. Download the current course CSV (75/150 µm).')
print('Rows and columns:', data.shape)
display(data[['cutter_id', 'cut_number', 'wear_mean_um', 'wear_level']].head(3))

## Required Checkpoint 1 — Define the labels

| Mean wear | Multiclass label | Binary label |
|---|---|---|
| ≤75 µm | 0: Slight-wear | 0: Normal wear |
| >75 and <150 µm | 1: Intermediate-wear | 0: Normal wear |
| ≥150 µm | 2: Severe-wear | 1: Severe wear |

**Binary positive class = Severe (1).** Multiclass 1 means Intermediate.
These course-defined categories are not industrial replacement limits.

![Wear classes and their binary mapping; course-defined 75 and 150 micrometre boundaries](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab05/wear_class_mapping_corrected.png)

### Checkpoint 1 response — apply the label definitions

Fill the numeric labels in this table; no paragraph is needed.

| Mean wear (µm) | Multiclass label (0/1/2) | Binary label (0/1) |
|---|---|---|
| 75 | TODO | TODO |
| 100 | TODO | TODO |
| 150 | TODO | TODO |

**Label 1 means:** multiclass = TODO; binary = TODO.

### Training, validation and test

Randomly split pooled c1/c4 cuts into 80% training and 20% validation, preserving
cutter × wear-class proportions (seed 42). All models use the same rows.
Nearby cuts can be correlated; c6 tests transfer to one other cutter.
Fit scaling on training only. After validation selects each model, refit its
scaler and model on c1/c4; use c6 only for evaluation. We classify current wear,
not future wear.

![Training, selection and final evaluation](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab05/evaluation_flow.png)

In [ ]:
# PROVIDED — RUN UNCHANGED. One split for all models and both tasks.
# True identifies a row from either development cutter.
development_rows = data['cutter_id'].isin(['c1', 'c4'])
# .loc selects those rows; .copy creates a separate working table.
development = data.loc[development_rows].copy()
# Keep all c6 rows apart until final evaluation.
test_rows = data['cutter_id'] == 'c6'
test = data.loc[test_rows].copy()
# Convert the class number to text, then form groups such as 'c1_0'.
class_text = development['wear_level'].astype(str)
strata = development['cutter_id'] + '_' + class_text
# Reserve 20% for validation. Seed 42 reproduces the same split.
# stratify keeps each cutter/class group represented in both partitions.
train, valid = train_test_split(development, test_size=0.20,
                               random_state=42, stratify=strata)
# Sorting only makes output readable; it does not change partition membership.
train = train.sort_index()
valid = valid.sort_index()
print('Train / validation / test rows:', len(train), len(valid), len(test))
print('Three-class counts: columns 0=Slight, 1=Intermediate, 2=Severe')
print('Training counts by cutter:')
display(pd.crosstab(train['cutter_id'], train['wear_level']))
print('Validation counts by cutter:')
display(pd.crosstab(valid['cutter_id'], valid['wear_level']))

### Prepare X, y and scaling

**X:** 11 sensor features per cut. **y:** the binary or multiclass wear label.
Wear measurements, cutter ID and cut number are excluded from X.
Fit scaling on training rows only; scaled features are unitless.

In [ ]:
# PROVIDED — RUN UNCHANGED. Explicit sensor inputs, identical to Lab 4.
sensor_features = [
    'force_x_mean', 'force_x_sd', 'force_y_mean', 'force_y_sd',
    'force_z_mean', 'force_z_sd', 'vibration_x_sd', 'vibration_y_sd',
    'vibration_z_sd', 'ae_rms_mean', 'ae_rms_sd',
]
X_train = train[sensor_features]  # Training rows, sensor columns only.
X_valid = valid[sensor_features]  # Validation rows, the same columns/order.
y_train_multi = train['wear_level']  # Original training labels: 0, 1 or 2.
y_valid_multi = valid['wear_level']  # Actual validation labels: 0, 1 or 2.
# Ask whether each cut has Severe wear; the result is True or False.
train_is_severe = y_train_multi == 2
valid_is_severe = y_valid_multi == 2
# Convert True to 1 and False to 0 for the binary task.
y_train_binary = train_is_severe.astype(int)
# Convert each Boolean decision into a class label: True becomes 1, False becomes 0.
y_valid_binary = valid_is_severe.astype(int)

scaler = StandardScaler()  # Create an unfitted scaling tool.
scaler.fit(X_train)  # Learn one mean and standard deviation per training column.
X_train_scaled = scaler.transform(X_train)  # Apply (value - mean) / SD.
X_valid_scaled = scaler.transform(X_valid)  # Reuse TRAINING means/SDs; do not fit again.
print('Training input shape:', X_train_scaled.shape)

## Required Checkpoint 2 — Metrics and binary classification

### Read the confusion matrix

Rows = **actual**; columns = **predicted**. Severe is positive.

| Actual / Predicted | Normal | Severe |
|---|---|---|
| Normal | TN: correct Normal | FP: false alarm |
| Severe | FN: missed Severe | TP: correct Severe |

Accuracy counts all correct predictions; it can hide missed Severe cases.

### Accuracy, precision, and recall practice

The figure uses **illustrative counts, not PHM results**: TN=8, FP=2, FN=3, TP=7.
Accuracy considers all predictions; precision considers **predicted Severe** cases;
recall considers **actual Severe** cases. F1 combines precision and recall.
**Before running the next cell**, calculate accuracy, precision and recall from
the counts and formulas below. Then fill the two precision/recall expressions
and run the cell to check your calculations. Accuracy code is supplied as an example.
`max(count, 1)` avoids division by zero;
this Lab reports undefined ratios as zero. Feedback shows PHM metrics separately.

![Illustrative counts and metric formulas without worked answers](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab05/classification_metrics_formulas.png)

In [ ]:
# REQUIRED — two expressions for the illustrative figure above, not the PHM matrix.
example_TN = 8
example_FP = 2
example_FN = 3
example_TP = 7
# COMPLETED EXAMPLE — accuracy counts correct predictions from BOTH classes.
example_correct = example_TP + example_TN
# Every example belongs to exactly one of TN, FP, FN or TP, so sum all four counts.
example_total = example_TP + example_TN + example_FP + example_FN
# Divide the number of correct predictions by the total number of examples.
example_accuracy = example_correct / example_total
print('Illustrative accuracy:', example_accuracy)  # Check after calculating it yourself.
# Division uses /. max(count, 1) avoids division by zero in this Lab's convention.
normal_recall = example_TN / max(example_TN + example_FP, 1)  # Completed analogous example.
# Precision considers predicted Severe cuts; recall considers actual Severe cuts.
severe_precision = None  # TODO: Use example_TP and example_FP; guard the denominator with max(..., 1).
# Among actual Severe cuts (TP + FN), count the ones detected as Severe (TP).
severe_recall = None  # TODO: Use example_TP and example_FN; guard the denominator with max(..., 1).
print('Normal recall:', normal_recall)
print('Your Severe precision and recall:', severe_precision, severe_recall)

In [ ]:
# PROVIDED — check the two expressions immediately after completing them.
metrics_ready = severe_precision is not None and severe_recall is not None
if metrics_ready:
    example_precision_check = example_TP / (example_TP + example_FP)
    example_recall_check = example_TP / (example_TP + example_FN)
    print('Illustrative precision / recall:', example_precision_check, example_recall_check)
    print('Your precision matches:', np.isclose(severe_precision, example_precision_check))
    print('Your recall matches:', np.isclose(severe_recall, example_recall_check))
    print('If either check is False, revise that expression and rerun both cells.')
else:
    print('Replace both None placeholders in the preceding REQUIRED cell, then rerun.')

### Checkpoint 2 response

Why do precision and recall use different denominators? One sentence is enough.

**Your response:** TODO

### Fit binary classifiers

**Baseline:** always predict the most common training class.
**Logistic Regression:** convert a linear score into a Severe-wear probability estimate.
In the figure, each $z_j$ is a scaled sensor feature; weights $w_j$ and intercept $b$
are learned from training data. No manual sigmoid calculation is required.
`fit` learns from training data;
`predict` returns class labels. Keep the supplied model settings.
In the illustration, predict Severe when $p \ge t$. Changing the probability
threshold $t$ changes labels without refitting; it does not change wear boundaries.

![Illustrative Logistic probability and thresholds](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab05/logistic_probability.png)

In [ ]:
# GUIDED — RUN UNCHANGED. Completed fit/predict examples.
binary_baseline = DummyClassifier(strategy='most_frequent')  # Always use the common class.
binary_baseline.fit(X_train_scaled, y_train_binary)  # Find the common TRAINING label.
baseline_binary_pred = binary_baseline.predict(X_valid_scaled)  # One label per validation cut.

# Create a model. C controls regularization; keep the supplied value.
# lbfgs is the fitting algorithm; max_iter limits its optimization iterations.
logistic_binary = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
logistic_binary.fit(X_train_scaled, y_train_binary)  # Learn from training inputs and labels.
logistic_binary_pred = logistic_binary.predict(X_valid_scaled)  # Predict unseen validation rows.
print('Baseline accuracy:', round(accuracy_score(y_valid_binary, baseline_binary_pred), 3))
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
print('Baseline Severe F1:', round(f1_score(y_valid_binary, baseline_binary_pred, zero_division=0), 3))

In [ ]:
# PROVIDED — RUN UNCHANGED. The matrices use the same validation rows.
binary_names = ['Normal wear', 'Severe wear']
# fig is the whole figure; axes[0] and axes[1] are its left and right panels.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Draw actual-vs-predicted counts. labels fixes numeric order; display_labels supplies names.
# ax chooses the panel; cmap chooses colors; colorbar=False omits the color scale bar.
ConfusionMatrixDisplay.from_predictions(y_valid_binary, baseline_binary_pred,
    labels=[0, 1], display_labels=binary_names, ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Majority baseline — validation')
ConfusionMatrixDisplay.from_predictions(y_valid_binary, logistic_binary_pred,
    labels=[0, 1], display_labels=binary_names, ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Logistic Regression — validation')
fig.tight_layout()  # Adjust spacing so labels fit.
plt.show()  # Display the completed figure.

# First argument: actual labels. Second: predicted labels. Fix the order to 0, 1.
binary_matrix = confusion_matrix(y_valid_binary, logistic_binary_pred, labels=[0, 1])
# Python counts positions from zero. [row, column] selects one matrix entry.
TN = int(binary_matrix[0, 0])  # Actual Normal, predicted Normal.
FP = int(binary_matrix[0, 1])  # Actual Normal, predicted Severe: false alarm.
FN = int(binary_matrix[1, 0])  # Actual Severe, predicted Normal: missed Severe.
TP = int(binary_matrix[1, 1])  # Actual Severe, predicted Severe.
print('TN, FP, FN, TP:', TN, FP, FN, TP)

### Add an SVM

An RBF SVM learns a nonlinear separating boundary, extending the SVR idea from
Lab 4 to classification. Keep `C=1.0` and `gamma='scale'` fixed.

Follow the Logistic example: fit on **X_train_scaled, y_train_binary**;
predict on **X_valid_scaled**. An SVM decision score is not a probability.
The two panels below use different synthetic datasets to illustrate boundary
shapes; they are not PHM results or a comparison of model performance.

![Synthetic two-feature SVM examples, not PHM results](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab05/svm_boundaries.png)

In [ ]:
# REQUIRED — two analogous expressions. Keep the supplied model settings.
# Create an RBF SVM. gamma sets kernel width; 'scale' derives it from training inputs.
# probability=False means we request class labels, not calibrated probabilities.
svm_binary = SVC(kernel='rbf', C=1.0, gamma='scale', probability=False)
# Follow Logistic Regression above: fit uses training inputs AND training labels.
svm_binary_fit = None  # TODO: Fit the supplied SVM using the binary training data.
# predict uses only validation inputs; actual labels are reserved for evaluation.
svm_binary_pred = None  # TODO: Predict binary labels for the validation inputs.

In [ ]:
# PROVIDED — check SVM completion and show actual PHM metrics separately.
binary_ready = svm_binary_fit is not None and svm_binary_pred is not None
if not binary_ready:
    print('Complete both SVM expressions; binary selection and final test will wait.')
print('Actual PHM Logistic validation metrics (not the illustrative counts):')
print('Severe precision:', precision_score(y_valid_binary, logistic_binary_pred, zero_division=0))
print('Severe recall:', recall_score(y_valid_binary, logistic_binary_pred, zero_division=0))

## Required Checkpoint 3 — Choose, predict, check

Choose Logistic Regression or RBF SVM by **validation Severe F1**; ties go to
Logistic Regression. Read the table, record your choice, then run confirmation.

In [ ]:
# PROVIDED — RUN UNCHANGED. Same rows, sensor inputs and preprocessing.
binary_selected = None
if binary_ready:
    # Calculate one number at a time. Actual labels come before predictions.
    # Binary precision/recall/F1 use Severe (label 1) as the positive class.
    # zero_division=0 reports zero when a metric's denominator is zero.
    baseline_accuracy = accuracy_score(y_valid_binary, baseline_binary_pred)
    logistic_accuracy = accuracy_score(y_valid_binary, logistic_binary_pred)
    svm_accuracy = accuracy_score(y_valid_binary, svm_binary_pred)
    baseline_precision = precision_score(y_valid_binary, baseline_binary_pred, zero_division=0)
    logistic_precision = precision_score(y_valid_binary, logistic_binary_pred, zero_division=0)
    svm_precision = precision_score(y_valid_binary, svm_binary_pred, zero_division=0)
    baseline_recall = recall_score(y_valid_binary, baseline_binary_pred, zero_division=0)
    logistic_recall = recall_score(y_valid_binary, logistic_binary_pred, zero_division=0)
    svm_recall = recall_score(y_valid_binary, svm_binary_pred, zero_division=0)
    # Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
    baseline_binary_f1 = f1_score(y_valid_binary, baseline_binary_pred, zero_division=0)
    # Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
    logistic_binary_f1 = f1_score(y_valid_binary, logistic_binary_pred, zero_division=0)
    # Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
    svm_binary_f1 = f1_score(y_valid_binary, svm_binary_pred, zero_division=0)
    # Create an empty table, then add columns. Keep the same model order throughout.
    binary_comparison = pd.DataFrame()
    binary_comparison['Model'] = ['Majority baseline', 'Logistic Regression', 'RBF SVM']
    binary_comparison['Accuracy'] = [baseline_accuracy, logistic_accuracy, svm_accuracy]
    binary_comparison['Severe precision'] = [baseline_precision, logistic_precision, svm_precision]
    binary_comparison['Severe recall'] = [baseline_recall, logistic_recall, svm_recall]
    binary_comparison['Severe F1'] = [baseline_binary_f1, logistic_binary_f1, svm_binary_f1]
    display(binary_comparison.round(3))  # Round displayed values only; keep original scores.

### Checkpoint 3 response — choose, predict, check

Before continuing:
- Binary model choice and validation Severe F1: TODO
- Raising the threshold from 0.50 to 0.75 makes the number of Severe predictions: TODO: increase / decrease / stay unchanged

After the threshold comparison, return here:
- FN changed from TODO to TODO. Explain why in one short sentence: TODO

An incorrect initial expectation is not penalized; check it against the results.


In [ ]:
# PROVIDED — compare this confirmation with your recorded choice.
if binary_ready:
    # Start with Logistic; change only if SVM is strictly better. A tie keeps Logistic.
    binary_selected = 'Logistic Regression'
    if svm_binary_f1 > logistic_binary_f1:
        binary_selected = 'RBF SVM'
    print('Binary selection:', binary_selected)

### Threshold practice — one example, one change

Run the supplied 0.50 example cell first. In the following **REQUIRED** cell,
change only `student_threshold` to **0.75**, then run the comparison.
Predict Severe when its probability is at least
the threshold. The model is already fitted; this changes predictions, not training.
Use the FP, FN and F1 table to complete CP3. Final testing keeps the default rule.


In [ ]:
# PROVIDED EXAMPLE — predict Severe using a threshold of 0.50.
class_probabilities = logistic_binary.predict_proba(X_valid_scaled)
severe_probability = class_probabilities[:, 1]  # Column 1 is P(Severe).
example_threshold = 0.50
# The comparison is True where the estimated positive-class probability reaches the cutoff.
pred_middle = (severe_probability >= example_threshold).astype(int)
# Count actual labels by row and predicted labels by column, in the specified label order.
matrix_middle = confusion_matrix(y_valid_binary, pred_middle, labels=[0, 1])
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
example_f1 = f1_score(y_valid_binary, pred_middle, zero_division=0)

In [ ]:
# REQUIRED — change only the threshold; keep the same model and probabilities.
student_threshold = 0.50  # TODO: Change 0.50 to 0.75 after recording your expectation.
# PROVIDED — True becomes 1 (Severe); False becomes 0 (Normal).
pred_high = (severe_probability >= student_threshold).astype(int)
# Count actual labels by row and predicted labels by column, in the specified label order.
matrix_high = confusion_matrix(y_valid_binary, pred_high, labels=[0, 1])
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
student_f1 = f1_score(y_valid_binary, pred_high, zero_division=0)

# Compare the two rules. Matrix [0, 1] counts FP; [1, 0] counts FN.
threshold_results = pd.DataFrame()
threshold_results['Threshold'] = [example_threshold, student_threshold]
threshold_results['False alarms (FP)'] = [matrix_middle[0, 1], matrix_high[0, 1]]
threshold_results['Missed Severe (FN)'] = [matrix_middle[1, 0], matrix_high[1, 0]]
threshold_results['Severe F1'] = [example_f1, student_f1]
display(threshold_results)

# A successful run does not mean the requested change has been completed.
if student_threshold != 0.75:
    print('CP3 is unfinished: change student_threshold to 0.75 and rerun this cell.')

<details>
<summary>Check your reasoning after running the comparison</summary>

A higher threshold cannot increase the number of Severe predictions. It can
reduce false alarms but increase missed Severe cases. Read the observed FN counts
from your table; the actual change depends on the predicted probabilities.
</details>

## Required Checkpoint 4 — Multiclass classification

Fit new models with labels 0/1/2, keeping inputs, rows and scaling unchanged.
Select by **macro-F1**, which weights each class equally:

$$\mathrm{Macro\ F1}=\frac{F_{1,0}+F_{1,1}+F_{1,2}}{3}.$$

In [ ]:
# GUIDED — RUN UNCHANGED. Repeat familiar fit/predict steps with three-class y.
# Start fresh models: the earlier binary models learned a different target.
multi_baseline = DummyClassifier(strategy='most_frequent')
multi_baseline.fit(X_train_scaled, y_train_multi)  # Find the common three-class training label.
baseline_multi_pred = multi_baseline.predict(X_valid_scaled)  # Predict that label for every row.

# Same model settings and scaled inputs; now labels can be 0, 1 or 2.
logistic_multi = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
logistic_multi.fit(X_train_scaled, y_train_multi)  # Learn from three-class training labels.
logistic_multi_pred = logistic_multi.predict(X_valid_scaled)  # Predict validation labels.

# Repeat the same three steps for the RBF SVM: create, fit, predict.
svm_multi = SVC(kernel='rbf', C=1.0, gamma='scale', probability=False)
svm_multi.fit(X_train_scaled, y_train_multi)  # Training data only.
svm_multi_pred = svm_multi.predict(X_valid_scaled)  # Validation inputs only.

In [ ]:
# PROVIDED — RUN UNCHANGED. Macro-F1 selects; other metrics provide context.
# labels fixes the evaluated classes. 'macro' gives the three class F1 scores equal weight.
baseline_multi_f1 = f1_score(y_valid_multi, baseline_multi_pred, labels=[0, 1, 2], average='macro', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
logistic_multi_f1 = f1_score(y_valid_multi, logistic_multi_pred, labels=[0, 1, 2], average='macro', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
svm_multi_f1 = f1_score(y_valid_multi, svm_multi_pred, labels=[0, 1, 2], average='macro', zero_division=0)
# Accuracy counts all correct predictions, regardless of their class.
baseline_multi_accuracy = accuracy_score(y_valid_multi, baseline_multi_pred)
logistic_multi_accuracy = accuracy_score(y_valid_multi, logistic_multi_pred)
svm_multi_accuracy = accuracy_score(y_valid_multi, svm_multi_pred)
# Assemble already-calculated values in the same model order for every column.
multi_comparison = pd.DataFrame()
multi_comparison['Model'] = ['Majority baseline', 'Logistic Regression', 'RBF SVM']
multi_comparison['Accuracy'] = [baseline_multi_accuracy, logistic_multi_accuracy, svm_multi_accuracy]
multi_comparison['Macro-F1'] = [baseline_multi_f1, logistic_multi_f1, svm_multi_f1]
display(multi_comparison.round(3))  # Display three decimal places without changing selection scores.

### Checkpoint 4 response — choose the multiclass model

Record the model with higher validation macro-F1; use Logistic Regression for a tie.

**Choice and macro-F1:** TODO

In [ ]:
# PROVIDED — check your recorded choice before evaluating c6.
# The predeclared rule uses macro-F1 only; exact ties keep Logistic Regression.
multi_selected = 'Logistic Regression'
if svm_multi_f1 > logistic_multi_f1:
    multi_selected = 'RBF SVM'
print('Multiclass selection:', multi_selected)

### Final evaluation — after recording your choices

Run the supplied code to refit each selected model and a fresh scaler on all
630 c1/c4 cuts, then evaluate c6. Keep the default prediction rule; do not retune
after seeing test results. Low test scores are valid findings.

In [ ]:
# PROVIDED — RUN UNCHANGED. Fresh fit after validation-based selection.
# These four expressions enable model evaluation; check the separate threshold edit for CP3.
final_ready = binary_ready and metrics_ready
if final_ready:
    X_refit = development[sensor_features]  # All 630 c1/c4 rows, same 11 input columns.
    X_test = test[sensor_features]  # All 315 c6 rows, same columns and order.
    refit_is_severe = development['wear_level'] == 2  # True for Severe wear.
    y_refit_binary = refit_is_severe.astype(int)  # True -> 1; False -> 0.
    y_refit_multi = development['wear_level']  # Keep original 0/1/2 labels.
    test_is_severe = test['wear_level'] == 2
    y_test_binary = test_is_severe.astype(int)  # Actual binary test labels for scoring.
    y_test_multi = test['wear_level']  # Actual three-class test labels for scoring.

    # Learn a NEW scaler from all development rows; never fit it on c6.
    final_scaler = StandardScaler()
    final_scaler.fit(X_refit)  # Learn means/SDs using 630 development rows.
    X_refit_scaled = final_scaler.transform(X_refit)  # Scale the refit inputs.
    X_test_scaled = final_scaler.transform(X_test)  # Reuse development means/SDs on c6.
    # Recreate the binary winner chosen on validation; do not choose again on test.
    final_binary_model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
    if binary_selected == 'RBF SVM':
        final_binary_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=False)
    final_binary_model.fit(X_refit_scaled, y_refit_binary)  # Fit the fresh winner on 630 cuts.
    final_binary_pred = final_binary_model.predict(X_test_scaled)  # Predict 315 test labels.

    # Independently recreate the multiclass winner using the three-class target.
    final_multi_model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
    if multi_selected == 'RBF SVM':
        final_multi_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=False)
    final_multi_model.fit(X_refit_scaled, y_refit_multi)  # Same refit inputs, different target.
    final_multi_pred = final_multi_model.predict(X_test_scaled)  # One 0/1/2 label per test cut.
else:
    print('Complete the metric and SVM expressions, then rerun in order. CP3 also requires the threshold edit.')

### Final evaluation: calculate test scores

In [ ]:
# PROVIDED — calculate test scores after fitting, without selecting models again.
if final_ready:
    # Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
    final_binary_f1 = f1_score(y_test_binary, final_binary_pred, zero_division=0)
    # Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
    final_multi_f1 = f1_score(y_test_multi, final_multi_pred, labels=[0, 1, 2], average='macro', zero_division=0)

### Compare validation and test

A strong validation score may not carry over to another cutter. These scores
describe different fits (504 training cuts vs 630 refit cuts) and different cutters.
No additional written response is required; do not retune using c6 results.

In [ ]:
# PROVIDED — one compact comparison; the two tasks use different F1 definitions.
if final_ready:
    selected_binary_valid = logistic_binary_pred
    if binary_selected == 'RBF SVM':
        selected_binary_valid = svm_binary_pred
    selected_multi_valid = logistic_multi_pred
    if multi_selected == 'RBF SVM':
        selected_multi_valid = svm_multi_pred
    final_comparison = pd.DataFrame()
    final_comparison['Task'] = ['Binary (Severe F1)', 'Multiclass (macro-F1)']
    final_comparison['Model'] = [binary_selected, multi_selected]
    # Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
    final_comparison['Validation F1'] = [f1_score(y_valid_binary, selected_binary_valid),
        # Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
        f1_score(y_valid_multi, selected_multi_valid, average='macro', zero_division=0)]
    final_comparison['c6 test F1'] = [final_binary_f1, final_multi_f1]
    # Round only the displayed table; model selection still uses the unrounded scores.
    display(final_comparison.round(3))

## Optional follow-up — ungraded

For optional self-study, see the [Practice index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Practice_index.ipynb).

## Submission and reproducibility checklist

Rendered checkboxes are not clickable; edit `[ ]` to `[x]` in this Markdown cell.

- [ ] Enter name and WSU AccessID.
- [ ] Complete five small code edits and CP1–CP4; confirm the threshold table shows 0.50 and 0.75.
- [ ] Record validation choices before final test.
- [ ] Restart and Run all; confirm required outputs are visible.
- [ ] Save `Lab05_Firstname_Lastname.ipynb` and a readable matching `.pdf`.
- [ ] Submit both through Canvas. No separate report.

Canvas controls due dates. PDF export issues receive feedback, not point deductions.

## References

- PHM Society, [2010 PHM Society Conference Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/).
- Course-derived feature table: [README](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md)
  and [data dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md).
- scikit-learn: [Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression),
  [SVM](https://scikit-learn.org/stable/modules/svm.html),
  [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).

Acknowledge the PHM Society source and the course-derived processing/labels
when reusing these data. Slight/Intermediate/Severe and the binary Normal
grouping are instructional categories, not official benchmark severity labels.

The original dataset license is not replaced by the course license. This previously studied corpus is not an untouched research holdout; one test cutter does not establish broad generalization.